In [1]:
%load_ext autoreload
%autoreload 2

In [45]:
from pathlib import Path
import sys
import os
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

from utils.seku_crd_corpus_utils import *
from prediction.predict_ner import *
import random

### Corpus Explorartion and Validation from S3 ###

**Check how many papers in S3**

This may take few minutes for the first run

In [3]:
print(AWS_BUCKET_NAME)
print(AWS_DIRECTORY)
stats = count_s3_files(bucket_name=AWS_BUCKET_NAME, directory=AWS_DIRECTORY)
print(stats)

psm-iiser
SEKU-CRD-v1/index_v1

S3 Directory: s3://psm-iiser/SEKU-CRD-v1/index_v1
JSON Files: 617551
Total Size: 105.34 GB
{'total_files': 617551, 'total_size_gb': 105.34}


**Load Main Papers (Paper that refer/cite other papers)**

In [4]:
paper_index = load_paper_index()
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 166310line [00:24, 6895.55line/s] 


Loaded 166310 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (6496.9 MB in memory)
Total 166310 Main papers


**Random Paper Pickup and validation**

In [5]:
rand_idx = random.randint(0,nb_main_papers-1)
rand_paper_id = all_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
nb_all_refs = len(main_data["bib_entry_ids"]) # [A] all unfiltered original references
nb_usable_refs = len(main_data["references"]) # [B] all references that have either an openalex or arxiv id, but does not ensure in-corpus membership
nb_inc_refs = len(main_data["in_corpus_references"]) # [C] all references that have either an openalex or arxiv id, and in-corpus membership
ratio = nb_inc_refs / nb_all_refs # [D] in corpus / original unfiltered
print(f'Title: {title}')
print(f"[Derived] All Refs [{nb_all_refs}],Usable Refs [{nb_usable_refs}], In-Corpus_Refs [{nb_inc_refs}], Ratio [{ratio:.2f}]")

# some values are precomputed in paper index, they should have same value as computed above
nb_all_refs_r = main_data["total_bib_entries"] # basically len(A)
nb_inc_refs_r = main_data["nb_reference_in_corpus"] # basically len(C)
ratio_r = main_data["reference_ratio"] # should be same as in D
print(f"[Precomputed] All Refs [{nb_all_refs_r}], In-Corpus_Refs [{nb_inc_refs_r}], Ratio [{ratio_r:.2f}]")

Random Main paper Index:123957 [Corpus ID: 1510.00494]
Title: Study of vacuum behavior for inert models with discrete $Z_{2}$-like and
  abelian $U(1)$ symmetries
[Derived] All Refs [83],Usable Refs [43], In-Corpus_Refs [27], Ratio [0.33]
[Precomputed] All Refs [83], In-Corpus_Refs [27], Ratio [0.33]


**Checking Paper Content from S3**

In [6]:
in_corpus_refs_ids = main_data["in_corpus_references"] # list of IDs of [C]
# we need to check if these ids truly exists in our Full Paper Data
# our full paper data is in S3 😊, so we will get it from there. 
results = await get_papers_from_s3(in_corpus_refs_ids)
found_full_data = 0
main_paper_data = get_paper_from_s3(rand_paper_id)
main_title,main_abstract = get_title_and_abstract(main_paper_data)
print(f"Checking refernces for Main paper {rand_paper_id}\n{main_title}")
print("="*5)
for paper_id, data in results.items():
    if data is not None:
        ref_title,ref_abstract = get_title_and_abstract(data)
        print(f"{paper_id} found 😁 \n {ref_title}")
        found_full_data += 1
    else:
        print(f"{paper_id} Not found ☹️")
full_data_ratio = found_full_data / nb_all_refs_r
print(f"{full_data_ratio*100:0.2f}% references have full data")

Fetching papers: 100%|██████████| 27/27 [00:01<00:00, 21.70it/s]


Checking refernces for Main paper 1510.00494
Study of vacuum behavior for inert models with discrete $Z_{2}$-like and
  abelian $U(1)$ symmetries
=====
hep-ph/9903289 found 😁 
 Mass bounds of the lightest CP-even Higgs boson in the two-Higgs-doublet
  model
hep-ph/9811234 found 😁 
 Vacuum Stability Bounds in the Two-Higgs Doublet Model
hep-ph/0201160 found 😁 
 Effective operators and vacuum instability as heralds of new physics
1307.5193 found 😁 
 Stability, Higgs Boson Mass and New Physics
hep-ph/0104016 found 😁 
 On the metastability of the Standard Model vacuum
1511.02879 found 😁 
 One-loop contributions to neutral minima in the inert doublet model
1205.6497 found 😁 
 Higgs mass and vacuum stability in the Standard Model at NNLO
1503.07078 found 😁 
 Inert scalars and vacuum metastability around the electroweak scale
hep-ph/9506284 found 😁 
 Dynamics of Two Higgs Doublet CP Violation and Baryogenesis at the
  Electroweak Phase Transition
1503.02135 found 😁 
 Search for a 'stable alig

### Model Prediction ###

In [7]:
LOCAL_MODEL_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/model"
WANDB_PROJECT = "psm-iiser-citation-relevancy"
WANDB_ARTIFACT = "qwen-ner-adapter"

In [8]:
tokenizer, model = load_ft_ner_model(local_dir=LOCAL_MODEL_DIR,wandb_project=WANDB_PROJECT,artifact_name=WANDB_ARTIFACT)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/parthosarothi/.netrc.


Looking up 'qwen-ner-adapter:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: ner-model/qwen-ner/100926-165108
  Downloading: ner-model/qwen-ner/100926-165108/adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/README.md
  Downloading: ner-model/qwen-ner/100926-165108/adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/adapter_config.json
  Downloading: ner-model/qwen-ner/100926-165108/adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/adapter_model.safetensors
  Downloading: ner-model/qwen-ner/100926-165108/adapter_model/added_tokens.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/added_tokens.json
  Downloading: ner-model/qwen-ner/100926-165108/adapter_model/chat_template.jinj

Loading weights: 100%|██████████| 339/339 [01:39<00:00,  3.40it/s]


**Take some samples of main papers**

In [9]:
sample = 25
paper_index = load_paper_index(max_files=sample) # load some papers
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 24line [00:00, 187.49line/s]


Loaded 25 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (0.9 MB in memory)
Total 25 Main papers


**Randomly pick a main paper from samples**

In [46]:
rand_idx = random.randint(0,sample-1)
rand_paper_id = all_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
print(f'Title: {title}')
main_paper_data = get_paper_from_s3(rand_paper_id)
print("Retrieved main paper data from S3...Extracting titles (not needed) and abstract")
main_title,main_abstract = get_title_and_abstract(main_paper_data)
print("Extracted title and abstract from raw data...Finding Named Entities")
result = predict_ner_ft(main_abstract,tokenizer,model,max_new_tokens=2048)
print("Entity Extraction Findinings complete")

Random Main paper Index:13 [Corpus ID: 0912.0010]
Title: The Nuts and Bolts of Einstein-Maxwell Solutions
Retrieved main paper data from S3...Extracting titles (not needed) and abstract
Extracted title and abstract from raw data...Finding Named Entities
Entity Extraction Findinings complete


In [47]:
print(main_abstract) # just check abstract for reference

We find new non-supersymmetric solutions of five-dimensional ungauged
supergravity coupled to two vector multiplets. The solutions are regular,
horizonless and have the same asymptotic charges as non-extremal charged black
holes. An essential ingredient in our construction is a four-dimensional
Euclidean base which is a solution to Einstein-Maxwell equations. We construct
stationary solutions based on the Euclidean dyonic Reissner-Nordstrom black
hole as well as a six-parameter family with a dyonic Kerr-Newman-NUT base.
These solutions can be viewed as compactifications of eleven-dimensional
supergravity on a six-torus and we discuss their brane interpretation.


**See Raw Entities by Label**

In [48]:
result_g = combine_entities_by_label(result)
for r in result_g:
    print(r)
    print("="*5)
    print(result_g[r])
    print("--"*5)

CONTRIBUTION
=====
new non-supersymmetric solutions|
stationary solutions|
six-parameter family
----------
METHOD
=====
five-dimensional ungauged
supergravity|
Einstein-Maxwell equations
----------
KEY_CONCEPT
=====
two vector multiplets|
four-dimensional
Euclidean base|
Euclidean dyonic Reissner-Nordstrom black
hole|
dyonic Kerr-Newman-NUT base|
compactifications of eleven-dimensional
supergravity on a six-torus|
brane interpretation
----------
FINDING
=====
regular,
horizonless and have the same asymptotic charges as non-extremal charged black
holes
----------


**See the Entities in a Context Window**

In [49]:
result_aug = augment_context_by_label(result,main_abstract,window=0)
for r in result_aug:
    print(r)
    print("="*5)
    for rt in result_aug[r]:
        print(rt["text"])
    print("--"*5)

CONTRIBUTION
=====
We find  <new non-supersymmetric solutions>  of five-dimensional ungauged supergravity coupled to two vector multiplets.
We construct  <stationary solutions>  based on the Euclidean dyonic Reissner-Nordstrom black hole as well as a  <six-parameter family>  with a dyonic Kerr-Newman-NUT base.
----------
METHOD
=====
We find  <new non-supersymmetric solutions>  of  <five-dimensional ungauged supergravity>  coupled to two vector multiplets.
An essential ingredient in our construction is a  <four-dimensional Euclidean base>  which is a solution to  <Einstein-Maxwell equations> .
----------
KEY_CONCEPT
=====
We find  <new non-supersymmetric solutions>  of  <five-dimensional ungauged supergravity>  coupled to  <two vector multiplets> .
An essential ingredient in our construction is a  <four-dimensional Euclidean base>  which is a solution to Einstein-Maxwell equations.
We construct  <stationary solutions>  based on the  <Euclidean dyonic Reissner-Nordstrom black hole>  as 

In [12]:
abstract = """With the rapid growth of Web-based academic publications, more
and more papers are being published annually, making it increasingly difficult to find relevant prior work. Citation prediction aims
to automatically suggest appropriate references, helping scholars
navigate the expanding scientific literature. Here we present CiteRAG, the first comprehensive retrieval-augmented generation
(RAG)-integrated benchmark for evaluating large language models
on academic citation prediction, featuring a multi-level retrieval
strategy, specialized retrievers, and generators. Our benchmark
makes four core contributions: (1) We establish two instances of the
citation prediction task with different granularity. Task 1 focuses on
coarse-grained list-specific citation prediction, while Task 2 targets
fine-grained position-specific citation prediction. To enhance these
two tasks, we build a dataset containing 7,267 instances for Task
1 and 8,541 instances for Task 2, enabling comprehensive evaluation of both retrieval and generation. (2) We construct a three-level
large-scale corpus with 554k papers spanning many major subfields,using an incremental pipeline. (3) We propose a multi-level hybrid
RAG approach to citation prediction, fine-tuning embedding models
with contrastive learning to capture complex citation relationships,
paired with specialized generation models. (4) We conduct extensive experiments across state-of-the-art language models, including
closed-source APIs, open-source models, and our fine-tuned generators, demonstrating the effectiveness of our framework. Our
open-source toolkit enables reproducible evaluation and focuses on
academic literature, providing the first comprehensive evaluation
framework for citation prediction and serving as a methodological
template for other scientific domains. Our source code and data are
released at https://github.com/LQgdwind/CiteRAG."""

result = predict_ner_ft(abstract,tokenizer,model)
print(result)

[{'text': 'Web-based academic publications', 'label': 'KEY_CONCEPT', 'start': 25, 'end': 56}, {'text': 'rapid growth', 'label': 'PROBLEM', 'start': 9, 'end': 21}, {'text': 'difficult to find relevant prior work', 'label': 'PROBLEM', 'start': 132, 'end': 169}, {'text': 'Citation prediction', 'label': 'OBJECTIVE', 'start': 171, 'end': 190}, {'text': 'automatically suggest appropriate references', 'label': 'OBJECTIVE', 'start': 199, 'end': 243}, {'text': 'scientific literature', 'label': 'KEY_CONCEPT', 'start': 285, 'end': 306}, {'text': 'CiteRAG', 'label': 'CONTRIBUTION', 'start': 324, 'end': 331}, {'text': 'retrieval-augmented generation\n(RAG)-integrated benchmark', 'label': 'CONTRIBUTION', 'start': 357, 'end': 414}, {'text': 'large language models', 'label': 'KEY_CONCEPT', 'start': 430, 'end': 451}, {'text': 'multi-level retrieval\nstrategy', 'label': 'CONTRIBUTION', 'start': 497, 'end': 527}, {'text': 'specialized retrievers', 'label': 'CONTRIBUTION', 'start': 529, 'end': 551}, {'tex

In [15]:
result_g = combine_entities_by_label(result)
for r in result_g:
    print(r)
    print("="*5)
    print(result_g[r])
    print("--"*5)

KEY_CONCEPT
=====
Web-based academic publications|
scientific literature|
large language models|
coarse-grained list-specific citation prediction|
fine-grained position-specific citation prediction|
two tasks|
complex citation relationships|
state-of-the-art language models
----------
PROBLEM
=====
rapid growth|
difficult to find relevant prior work
----------
OBJECTIVE
=====
Citation prediction|
automatically suggest appropriate references
----------
CONTRIBUTION
=====
CiteRAG|
retrieval-augmented generation
(RAG)-integrated benchmark|
multi-level retrieval
strategy|
specialized retrievers|
generators|
two instances of the
citation prediction task|
three-level
large-scale corpus|
multi-level hybrid
RAG approach|
specialized generation models
----------
DATASET
=====
dataset
----------
METRIC
=====
7,267 instances|
8,541 instances|
554k papers
----------
METHOD
=====
retrieval|
generation|
incremental pipeline|
contrastive learning
----------
